In [3]:
import os
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

In [4]:
import json
import time
import copy
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


In [5]:
# Paths
DATA_DIR   = Path('/content/drive/MyDrive/characters_train')
MODEL_PATH = Path('model.pth')

# Image settings
IMG_SIZE  = 64          # resize both sides to IMG_SIZE × IMG_SIZE

BATCH_SIZE    = 64
EPOCHS        = 40
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
VAL_SPLIT     = 0.15     # 15 % of training data used for validation
PATIENCE      = 8        # early-stopping patience (epochs)
LR_PATIENCE   = 4        # ReduceLROnPlateau patience

In [6]:

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),
                             scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])


# our dataset class
class SimpsonsDataset(Dataset):
    """
    Reads images from a directory tree:
        root/
          class_A/img1.jpg ...
          class_B/img1.jpg ...
    """
    def __init__(self, root: Path, transform=None):
        self.transform = transform
        self.samples   = []   # list of (path, label_idx)
        self.classes   = sorted([d.name for d in root.iterdir()
                                  if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
        for cls in self.classes:
            cls_dir = root / cls
            for img_path in cls_dir.iterdir():
                if img_path.suffix.lower() in valid_ext:
                    self.samples.append((img_path,
                                         self.class_to_idx[cls]))

        print(f'Found {len(self.samples)} images across '
              f'{len(self.classes)} classes.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# Build splits
full_dataset = SimpsonsDataset(DATA_DIR, transform=None)   # transforms set later

n_total = len(full_dataset)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(
    full_dataset, [n_train, n_val], generator=generator
)

# Wrap subsets so each gets its own transform
class TransformWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img_path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(img_path).convert('RGB')
        return self.transform(img), label

train_ds = TransformWrapper(train_subset, train_transforms)
val_ds   = TransformWrapper(val_subset,   val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

NUM_CLASSES = len(full_dataset.classes)
print(f'Train: {n_train} | Val: {n_val} | Classes: {NUM_CLASSES}')
print('Classes:', full_dataset.classes)

Found 16795 images across 42 classes.
Train: 14276 | Val: 2519 | Classes: 42
Classes: ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson', 'carl_carlson', 'charles_montgomery_burns', 'chief_wiggum', 'cletus_spuckler', 'comic_book_guy', 'disco_stu', 'edna_krabappel', 'fat_tony', 'gil', 'groundskeeper_willie', 'homer_simpson', 'kent_brockman', 'krusty_the_clown', 'lenny_leonard', 'lionel_hutz', 'lisa_simpson', 'maggie_simpson', 'marge_simpson', 'martin_prince', 'mayor_quimby', 'milhouse_van_houten', 'miss_hoover', 'moe_szyslak', 'ned_flanders', 'nelson_muntz', 'otto_mann', 'patty_bouvier', 'principal_skinner', 'professor_john_frink', 'rainier_wolfcastle', 'ralph_wiggum', 'selma_bouvier', 'sideshow_bob', 'sideshow_mel', 'snake_jailbird', 'troy_mcclure', 'waylon_smithers']


In [7]:
class ConvBlock(nn.Module):
    """Conv → BN → ReLU (→ optional MaxPool)"""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class SpringfieldCNN(nn.Module):
    """
    A compact CNN designed from scratch for Simpsons character classification.

    Input : (B, 3, 64, 64)
    Output: (B, NUM_CLASSES)

    Feature-map size progression (64 → 32 → 16 → 8 → 4 → GAP → FC)
    """
    def __init__(self, num_classes: int):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1  → 32×32
            ConvBlock(3,   32,  pool=True),
            nn.Dropout2d(0.1),

            # Block 2  → 16×16
            ConvBlock(32,  64,  pool=True),
            nn.Dropout2d(0.1),

            # Block 3  → 8×8
            ConvBlock(64,  128, pool=True),
            nn.Dropout2d(0.2),

            # Block 4  → 4×4
            ConvBlock(128, 256, pool=True),
            nn.Dropout2d(0.2),

            # Block 5  → 2×2  (no pool)
            ConvBlock(256, 256, pool=False),
        )

        # Global Average Pooling squeezes spatial dims to 1×1
        self.gap = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,
                                        mode='fan_out',
                                        nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


model = SpringfieldCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Quick param count
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters : {total_params:,}')
print(f'Trainable params : {train_params:,}')
print(model)

Total parameters : 2,458,506
Trainable params : 2,458,506
SpringfieldCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): Dropout2d(p=0.1, inplace=False)
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Con

In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(model.parameters(),
                         lr=LR,
                         weight_decay=WEIGHT_DECAY)

# Warm-up for 5 epochs, then cosine decay
def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    return 1.0

warmup_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - 5, eta_min=1e-6
)


In [9]:
def run_epoch(loader, model, criterion, optimizer=None, phase='train'):
    #One full pass through `loader`. Returns (avg_loss, accuracy)
    is_train = (phase == 'train')
    model.train(is_train)

    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total


# History tracking
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'lr': []
}

best_val_acc   = 0.0
best_weights   = None
patience_count = 0

print(f'Starting training for up to {EPOCHS} epochs on {DEVICE}\n')
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    ep_start = time.time()

    train_loss, train_acc = run_epoch(train_loader, model, criterion,
                                      optimizer, phase='train')
    val_loss,   val_acc   = run_epoch(val_loader,   model, criterion,
                                      phase='val')

    # Step the correct scheduler
    current_lr = optimizer.param_groups[0]['lr']
    if epoch <= 5:
        warmup_scheduler.step()
    else:
        cosine_scheduler.step()

    # Record
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    ep_time = time.time() - ep_start
    print(f'Epoch [{epoch:>3}/{EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc*100:.2f}%  |  '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc*100:.2f}%  |  '
          f'LR: {current_lr:.6f}  [{ep_time:.1f}s]')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = copy.deepcopy(model.state_dict())
        patience_count = 0
        print(f'  ✓ New best val accuracy: {best_val_acc*100:.2f}%  → model saved')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\nEarly stopping triggered after {epoch} epochs.')
            break

total_time = time.time() - start_time
print(f'\nTraining complete in {total_time/60:.1f} min.')
print(f'Best validation accuracy: {best_val_acc*100:.2f}%')

Starting training for up to 40 epochs on cuda

Epoch [  1/40]  Train Loss: 3.6301  Acc: 7.39%  |  Val Loss: 3.1007  Acc: 16.59%  |  LR: 0.000200  [3626.9s]
  ✓ New best val accuracy: 16.59%  → model saved
Epoch [  2/40]  Train Loss: 3.1732  Acc: 16.90%  |  Val Loss: 2.7205  Acc: 33.78%  |  LR: 0.000400  [91.1s]
  ✓ New best val accuracy: 33.78%  → model saved
Epoch [  3/40]  Train Loss: 2.8656  Acc: 27.96%  |  Val Loss: 2.4055  Acc: 44.62%  |  LR: 0.000600  [93.0s]
  ✓ New best val accuracy: 44.62%  → model saved
Epoch [  4/40]  Train Loss: 2.5586  Acc: 38.81%  |  Val Loss: 2.0914  Acc: 54.59%  |  LR: 0.000800  [92.7s]
  ✓ New best val accuracy: 54.59%  → model saved
Epoch [  5/40]  Train Loss: 2.2933  Acc: 47.95%  |  Val Loss: 1.8847  Acc: 59.47%  |  LR: 0.001000  [94.2s]
  ✓ New best val accuracy: 59.47%  → model saved
Epoch [  6/40]  Train Loss: 2.0773  Acc: 56.14%  |  Val Loss: 1.7071  Acc: 65.98%  |  LR: 0.001000  [93.1s]
  ✓ New best val accuracy: 65.98%  → model saved
Epoch [  7